# Lab: Build Your First RAG System (Using PDFs)

In this lab students will build a **Retrieval Augmented Generation (RAG)** system from scratch.

What you will learn:
1. Load data from PDFs
2. Split documents into chunks
3. Create embeddings
4. Store embeddings in a vector database
5. Retrieve relevant chunks
6. Send context to an LLM to answer questions

Architecture:

PDF → Chunking → Embeddings → Vector DB → Similarity Search → LLM


# Step 1 — Install Required Libraries

In [7]:
!pip install langchain faiss-cpu pypdf sentence-transformers openai langchain-community langchain-text-splitters


  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 18.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 21.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 30.0 MB/s  0:00:00
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)

   ----------------------------------------  0/20 [python-dotenv]
   -- -------------------------------------  1/20 [propcache]
   ------ ---------------------------------  3/20 [multidict]
   -------- -------------------------------  4/20 [marshmallow]
   ------------ ---------------------------  6/20 [greenlet]
   ------------ ---------------------------  6/20 [greenlet]
   -------------- -------------------------  7/20 [frozenlist]
   -


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Step 2 — Import Libraries

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from openai import OpenAI
import os

c:\Projects\PVS1\Project1-PVS\Project1-PVS\rag-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Step 3 — Load a PDF Document

Place a PDF file in the same folder as this notebook.

Example: `policy.pdf`

In [9]:
loader = PyPDFLoader('eticket.pdf')
documents = loader.load()

print('Number of pages loaded:', len(documents))

Number of pages loaded: 2


# Step 4 — Split the Document into Chunks

Large documents must be split into smaller pieces before creating embeddings.

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print('Number of chunks created:', len(chunks))

Number of chunks created: 6


# Step 5 — Create Embeddings

We will use a **free embedding model** from Sentence Transformers.

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2'
)

C:\Users\hetarra\AppData\Local\Temp\ipykernel_66632\1235873231.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8749.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Step 6 — Create a Vector Database

We will use **FAISS**, a fast local vector database.

In [12]:
vector_db = FAISS.from_documents(chunks, embeddings)

print('Vector database created successfully')

Vector database created successfully


In [17]:
print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1}:")
    print(chunk.page_content)

Total chunks: 6

Chunk 1:
Trip	
ID	:	
251115634052
Baggage	(per	Adult/Child)	–
Check-in:	15kg	(1	piece),	Cabin:	7kg	(1	piece)
Mr	Hemanth	Tarra
6,732
885
50
721
-995
342
7,735
Hyderabad	to	Kochi	
Sat,	29	Nov	2025
IndiGo
6E	-	6682	
Fare	type:	
REGULAR
HYD	
14:15
15:55	
COK
Sat,	29	Nov	2025
1h	40min
Sat,	29	Nov	2025
Hyderabad	-	
Rajiv	Gandhi
International
Economy
Kochi	-	
Cochin	International	Airport
TRAVELLERS
AIRLINE	PNR
TICKET	NO.
W8T3JG
W8T3JG
TIP:
	
Learn	the	three	letter	code	of	your	airport.	It	helps	in	several	ways!

Chunk 2:
W8T3JG
W8T3JG
TIP:
	
Learn	the	three	letter	code	of	your	airport.	It	helps	in	several	ways!
ABOUT	THIS	TRIP
The	condition	of	carriage	(COC)	for	IndiGo	can	be	found	at	
https://www.goindigo.in/information/conditions-of-carriage.html
Note:	Except	for	medical	
devices,	electronic	devices	which	are	larger	than	a	
cell	phone/smart	phone	cannot	be	carried	in	the	
cabin	of	the	aircraft.
Note:	Only	cell	phones	
of	dimension	Length:	16cm	x	Width:	9.3cm	x	Depth:

Chunk

In [18]:
vector_db.save_local("my_faiss_index")

# Step 7 — Ask a Question

In [1]:
query = 'What is this document about?'

# Step 8 — Retrieve Relevant Chunks

In [14]:
results = vector_db.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print('---')

W8T3JG
W8T3JG
TIP:
	
Learn	the	three	letter	code	of	your	airport.	It	helps	in	several	ways!
ABOUT	THIS	TRIP
The	condition	of	carriage	(COC)	for	IndiGo	can	be	found	at	
https://www.goindigo.in/information/conditions-of-carriage.html
Note:	Except	for	medical	
devices,	electronic	devices	which	are	larger	than	a	
cell	phone/smart	phone	cannot	be	carried	in	the	
cabin	of	the	aircraft.
Note:	Only	cell	phones	
of	dimension	Length:	16cm	x	Width:	9.3cm	x	Depth:
---
cabin	of	the	aircraft.
Note:	Only	cell	phones	
of	dimension	Length:	16cm	x	Width:	9.3cm	x	Depth:	
1.5cm	are	allowed	in	the	cabin	baggage.	Please	carry	
all	other	electronic
equipment	inside	check-in	
baggage.
Use	your	Trip	ID	for	all	
communication	with	Cleartrip	about	this	booking
Please	reach	the	airport	
3	hours	
before	the	departure	time.	
Check-in	counters	at	the	airport	close	
60	minutes	before	departure
Your	carry-on	baggage	
shouldn't	weigh	more	than	7kgs
Carry	photo	identification,
---
with	Cleartrip	instead	of	doing	so	dire

# Step 9 — Send Context to the LLM

In [15]:
context = '\n'.join([doc.page_content for doc in results])

prompt = f"""
Use the context below to answer the question.

Context:
{context}

Question:
{query}
"""

print(prompt)


Use the context below to answer the question.

Context:
W8T3JG
W8T3JG
TIP:
	
Learn	the	three	letter	code	of	your	airport.	It	helps	in	several	ways!
ABOUT	THIS	TRIP
The	condition	of	carriage	(COC)	for	IndiGo	can	be	found	at	
https://www.goindigo.in/information/conditions-of-carriage.html
Note:	Except	for	medical	
devices,	electronic	devices	which	are	larger	than	a	
cell	phone/smart	phone	cannot	be	carried	in	the	
cabin	of	the	aircraft.
Note:	Only	cell	phones	
of	dimension	Length:	16cm	x	Width:	9.3cm	x	Depth:
cabin	of	the	aircraft.
Note:	Only	cell	phones	
of	dimension	Length:	16cm	x	Width:	9.3cm	x	Depth:	
1.5cm	are	allowed	in	the	cabin	baggage.	Please	carry	
all	other	electronic
equipment	inside	check-in	
baggage.
Use	your	Trip	ID	for	all	
communication	with	Cleartrip	about	this	booking
Please	reach	the	airport	
3	hours	
before	the	departure	time.	
Check-in	counters	at	the	airport	close	
60	minutes	before	departure
Your	carry-on	baggage	
shouldn't	weigh	more	than	7kgs
Carry	photo	identi

# Step 10 — Call GitHub Models

In [16]:
client = OpenAI(
    base_url='https://models.inference.ai.azure.com',
    api_key=os.getenv('GITHUB_TOKEN')
)

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': prompt}]
)

print(response.choices[0].message.content)

This document provides information related to a flight trip booked with IndiGo Airlines through Cleartrip. It includes details about the condition of carriage, baggage restrictions, check-in procedures, fare breakdown, and contact information for customer support. Key points include the allowance for only small electronic devices in the cabin, the weight limit for carry-on baggage, the need for photo identification, and recommendations for arriving at the airport well ahead of departure.


# Summary

You built a full **RAG pipeline**:

1. Loaded PDFs
2. Split text into chunks
3. Created embeddings
4. Stored them in FAISS
5. Retrieved relevant chunks
6. Used an LLM to answer questions

This is the foundation of modern AI knowledge assistants.